# 03 — Model-Consistent Structured FDI

本节构造经典线性模型中的结构化教学攻击 $a=Hc$，验证估计状态发生偏移但残差保持不变。

In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
from src.grid_model import build_three_bus_model, generate_measurement
from src.state_estimator import wls_estimate
from src.attack_generator import structured_fdi_attack

rng = np.random.default_rng(42)
model = build_three_bus_model(sigma=0.01)

z, _ = generate_measurement(model, rng)
normal = wls_estimate(z, model.H, model.R)

c = np.array([0.005, -0.004])
z_attack, a = structured_fdi_attack(z, model.H, c)
attacked = wls_estimate(z_attack, model.H, model.R)

print("Chosen state offset c =", c)
print("Attack vector a = Hc =", a)
print("Normal x_hat =", normal.x_hat)
print("Attacked x_hat =", attacked.x_hat)
print("x_hat^a - x_hat =", attacked.x_hat - normal.x_hat)


Chosen state offset c = [ 0.005 -0.004]
Attack vector a = Hc = [-0.05   0.02   0.072  0.122 -0.092]
Normal x_hat = [-0.03988588 -0.06093575]
Attacked x_hat = [-0.03488588 -0.06493575]
x_hat^a - x_hat = [ 0.005 -0.004]


In [3]:
residual_difference = np.linalg.norm(attacked.residual - normal.residual)

print("Normal residual =", normal.residual)
print("Attacked residual =", attacked.residual)
print("||r_attack - r_normal||_2 =", residual_difference)
print("Normal J =", normal.J)
print("Attacked J =", attacked.J)


Normal residual = [ 0.00418841 -0.01507857 -0.00089444 -0.00013454 -0.00643267]
Attacked residual = [ 0.00418841 -0.01507857 -0.00089444 -0.00013454 -0.00643267]
||r_attack - r_normal||_2 = 1.4946834900704541e-16
Normal J = 2.8710337100468237
Attacked J = 2.871033710046825


## Why this happens

Because

\[
z^a=z+Hc,
\]

the ideal linear WLS estimator shifts to

\[
\hat{x}^a=\hat{x}+c.
\]

Therefore

\[
r^a=z^a-H\hat{x}^a
   =z+Hc-H(\hat{x}+c)
   =z-H\hat{x}
   =r.
\]

这就是本教学仓库最核心的数学现象。


### 课堂任务

尝试不同的 `c`，并验证 `x_hat^a - x_hat ≈ c` 与 `r^a ≈ r`。